# DOSE — YOLOv12n Training on VAIPE_PILL

Train YOLOv12n (2.6M params) on 11,523 Vietnamese pill images (108 classes).

Expected time: ~1.5 hours for 100 epochs on P100.

In [ ]:
# Install ultralytics
!pip install -q ultralytics

In [ ]:
from ultralytics import YOLO
import os, shutil, json, random, glob
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

DATASET_PATH = "/kaggle/input/dose-yolo-dataset"

# Load class names
with open(os.path.join(DATASET_PATH, "class_names.json")) as f:
    class_names_dict = json.load(f)  # "0" -> "paracetamol 500mg"
class_names_list = [class_names_dict[str(i)] for i in range(108)]
print(f"Loaded {len(class_names_list)} class names")

# Patch YAML path
yaml_path = os.path.join(DATASET_PATH, "dataset.yaml")
with open(yaml_path) as f:
    content = f.read()
content = content.replace("path: /home/lducc/code/dose/data/yolo", f"path: {DATASET_PATH}")

working_yaml = "/kaggle/working/dataset.yaml"
with open(working_yaml, "w") as f:
    f.write(content)
print("Dataset ready:", working_yaml)


In [ ]:
model = YOLO("yolo12n.pt")

results = model.train(
    data=working_yaml,
    epochs=100,
    imgsz=640,
    batch=32,
    patience=20,
    device=0,
    name="dose_yolo12n",
    exist_ok=True,
)

In [ ]:
model.export(format="onnx", imgsz=640, half=True)

In [ ]:
onnx_src = "/kaggle/working/runs/detect/dose_yolo12n/weights/best.onnx"
onnx_dst = "/kaggle/working/vaipe12n.onnx"

if os.path.exists(onnx_src):
    shutil.copy(onnx_src, onnx_dst)
    print(f"ONNX saved: {onnx_dst}")
    print(f"Size: {os.path.getsize(onnx_dst) / 1e6:.1f} MB")
else:
    import glob
    found = glob.glob("/kaggle/working/runs/**/best.onnx", recursive=True)
    if found:
        shutil.copy(found[0], onnx_dst)
        print(f"Found: {found[0]} -> {onnx_dst}")
    else:
        print("ONNX not found. Check runs/ directory.")

In [ ]:
import glob
csv_files = glob.glob("/kaggle/working/runs/detect/dose_yolo12n/results.csv")
for f in csv_files:
    dst = os.path.join("/kaggle/working", os.path.basename(f))
    shutil.copy(f, dst)
    print(f"Copied: {dst}")

In [ ]:
# Benchmark: validation metrics
model = YOLO("/kaggle/working/runs/detect/dose_yolo12n/weights/best.pt")
metrics = model.val(data=working_yaml, split="val")

print(f"mAP50:     {metrics.box.map50:.3f}")
print(f"mAP50-95:  {metrics.box.map:.3f}")
print(f"Precision: {metrics.box.mp:.3f}")
print(f"Recall:    {metrics.box.mr:.3f}")

# Per-class metrics CSV
df = pd.DataFrame({
    "class_id": range(108),
    "name": class_names_list,
    "precision": metrics.box.p.tolist(),
    "recall": metrics.box.r.tolist(),
    "mAP50": metrics.box.ap50.tolist() if hasattr(metrics.box, "ap50") else [0]*108,
})
per_class_path = "/kaggle/working/per_class_metrics.csv"
df.to_csv(per_class_path, index=False)
print(f"Saved: {per_class_path}")


In [ ]:
# Full val inference: score every image
val_dir = Path(DATASET_PATH) / "images" / "val"
val_images = sorted(val_dir.glob("*.jpg"))
print(f"Running inference on {len(val_images)} validation images...")

results_all = model(list(map(str, val_images)), verbose=False)

image_scores = []
for img_path, r in zip(val_images, results_all):
    confs = r.boxes.conf.cpu().numpy() if len(r.boxes) > 0 else np.array([0.0])
    image_scores.append({
        "path": img_path,
        "avg_conf": float(np.mean(confs)),
        "n_boxes": len(r.boxes),
        "confs": confs,
    })

# Select 16 random + 16 worst-performing (lowest avg conf)
random.seed(42)
random_16 = random.sample(image_scores, 16)
worst_16 = sorted(image_scores, key=lambda x: x["avg_conf"])[:16]

print(f"Random samples avg confidence: {np.mean([s['avg_conf'] for s in random_16]):.3f}")
print(f"Worst samples avg confidence: {np.mean([s['avg_conf'] for s in worst_16]):.3f}")


In [ ]:
# Visualize: 4x4 grid of random samples
def plot_grid(samples, title, save_path):
    fig, axes = plt.subplots(4, 4, figsize=(20, 20))
    fig.suptitle(title, fontsize=16, y=1.02)
    for ax, s in zip(axes.flatten(), samples):
        r = model(str(s["path"]), verbose=False)[0]
        plotted = r.plot(labels=True, conf=True)
        ax.imshow(cv2.cvtColor(plotted, cv2.COLOR_BGR2RGB))
        ax.set_title(f"{s['path'].name}\nconf={s['avg_conf']:.2f}, n={s['n_boxes']}", fontsize=8)
        ax.axis("off")
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved: {save_path}")

plot_grid(random_16, "16 Random Validation Samples", "/kaggle/working/inference_random.png")


In [ ]:
# Visualize: 4x4 grid of worst-performing samples
plot_grid(worst_16, "16 Worst-Performing Samples (Lowest Avg Confidence)", "/kaggle/working/inference_worst.png")
